# <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

### References

- [Menon, S. et al. From electrons to phase diagrams with machine learning potentials using pyiron based automated workflows. npj Comput Mater 10, 261 (2024)](https://www.nature.com/articles/s41524-024-01441-0)
- [Menon, S., Lysogorskiy, Y., Rogal, J. & Drautz, R. Automated free-energy calculation from atomistic simulations. Phys. Rev. Materials 5, 103801 (2021)](https://link.aps.org/doi/10.1103/PhysRevMaterials.5.103801)
- [Poul, M., Huber, L. & Neugebauer, J. Automated Generation of Structure Datasets for Machine Learning Potentials and Alloys. Preprint](https://doi.org/10.21203/rs.3.rs-4732459/v1) (2024)


In this notebook, we will use the interatomic potentials for calculation of thermodynamic properties such as Helmholtz and Gibbs free energies, which in turn can be used for the calculation of phase diagrams. We will discuss [calphy](https://calphy.org/), the tool for automated calculation of free energies, and the methology involved. Furthermore, we will also discuss [Landau](https://github.com/eisenforschung/landau) which can be used to arrive at phase diagrams from free energies.

## <font style="font-family:roboto;color:#455e6c"> A simple phase diagram </font> 

<img src="img/phase_dia_1.png" width="40%">

Phase diagrams provide a wealth of information such as: coexisting lines, melting temperature, phase stability, nucleation mechanism.

## <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams: the essential ingredients </font> 

<img src="img/cimg4.png" width="30%">

Phase diagrams can be evaluated from free energy diagrams. The required input are:
- $G(P, T)$ for unary systems
- $G(x, T)$ for binary systems

To calculate this, we employ thermodynamic integration, in conjuction with a number of methodological improvements.

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Theory behind free energy calculation </b></p>
Please see the end of this notebook, or  <a href="https://journals.aps.org/prmaterials/abstract/10.1103/PhysRevMaterials.5.103801">this publication</a> for a longer discussion on the theory and algorithms.
</div>

Due to our limited time, we will show the calculation of melting temperature, the end point on the phase diagram. We leverage a software tool for calculation of free energies, [calphy](https://calphy.org) along with pyiron for efficient and automated free energy calculations. For a detailed tutorial, see [here](https://workshop.pyiron.org/potentials-workshop-2022/phase_diagram/Intro.html).

In [1]:
from core import Workflow
from core.gui import PyironFlow

## <font style="font-family:roboto;color:#455e6c"> Calculate free energy with temperature for solid</font> 

By now we have seen how the GUI works. We try to calculate the free energy of an Al FCC structure.

Helpful nodes:
- `atomistic -> structure -> build -> Bulk` to create a structure. Make sure to check `cubic` box.
- `atomistic -> structure -> transform -> Repeat` to create a supercell. A 3x3x3 cell should work for this example.
- `atomistic -> thermodynamics -> calphy -> InputClass` to set inputs for free energy calculation.
- `atomistic -> thermodynamics -> calphy -> SolidFreeEnergyWithTemp` for calculating the free energy. A good potential which is fast to try out this calculation would be `2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1`
- `atomistic -> thermodynamics -> calphy -> PlotFreeEnergy` for a plot.

In [2]:
from pyiron_nodes.atomistic.engine.lammps import ListPotentials
from pyiron_nodes.atomistic.calculator.calphy import InputClass

In [3]:
from pyiron_nodes.atomistic.calculator.calphy import SolidFreeEnergy, SolidFreeEnergyWithTemp, LiquidFreeEnergy, LiquidFreeEnergyWithTemp, PlotFreeEnergy, CalcPhaseTransformationTemp, CollectResults
from pyiron_nodes.atomistic.structure.build import Bulk
from pyiron_nodes.atomistic.structure.transform import Repeat


In [4]:
wf = Workflow("calphy_fe1")

In [5]:
wf.Bulk = Bulk(name="Al", cubic=True)
wf.Repeat = Repeat(structure=wf.Bulk, repeat_scalar=3)
wf.InputClass = InputClass()
wf.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(inp=wf.InputClass, structure=wf.Repeat, potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
wf.PlotFreeEnergy = PlotFreeEnergy(temperature=wf.SolidFreeEnergyWithTemp.outputs.temperature, free_energy=wf.SolidFreeEnergyWithTemp.outputs.free_energy)

In [6]:
pf = PyironFlow([wf], nodes_path="pyiron_nodes")
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Need the solution?</b></p>
Copy paste the following code into a cell and execute:
</br>
</br>
pf = PyironFlow([Workflow("calphy-fe1")]) </br>
pf.gui

</div>

## <font style="font-family:roboto;color:#455e6c"> Calculate melting temperature </font> 

Now we take a step forward and calculate the melting temperature. The melting temperature is the temperature at which the free energies of the solid and liquid phases are equal. 

In addition to the nodes above, the following would be useful:
- The steps for solid free energy are same as above, but with good temperature window to be set in the `InputClass` would be 800-1100 K.
- For liquid, we need to create a structure first. There are two ways to do it:
    - either check the `melting_cycle` box in `InputClass`
    - or use `atomistic -> structure -> transform -> Rattle` with `stdev` of approximately 0.6 to create a structure, similar to liquid. Hint: Use the `Plot3D` node to see how it looks like!
- `atomistic -> thermodynamics -> calphy -> LiquidFreeEnergyWithTemp` for calculating the free energy of the liquid phase
- A good temperature window to be set in the `InputClass` would be 800-1100 K.

In [7]:
from pyiron_nodes.atomistic.structure.transform import Rattle
from pyiron_nodes.thermodynamics.landau import TemperatureLinePhase, TransitionTemperature

In [8]:
wf = Workflow("calphy_tm1")

In [9]:
wf.Bulk = Bulk(name="Al", cubic=True)
wf.Repeat = Repeat(structure=wf.Bulk, repeat_scalar=3)
wf.Rattle = Rattle(structure=wf.Repeat, seed=42, stdev=0.6)
wf.InputClass = InputClass(temperature=900, temperature_stop=1100)
wf.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(inp=wf.InputClass, structure=wf.Repeat, potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
wf.LiquidFreeEnergyWithTemp = LiquidFreeEnergyWithTemp(inp=wf.InputClass, structure=wf.Rattle, potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
wf.Solid = TemperatureLinePhase(name="AlSolid", concentration=0, temperatures=wf.SolidFreeEnergyWithTemp.outputs.temperature, free_energies=wf.SolidFreeEnergyWithTemp.outputs.free_energy)
wf.Liquid = TemperatureLinePhase(name="AlLiquid", concentration=0, temperatures=wf.LiquidFreeEnergyWithTemp.outputs.temperature, free_energies=wf.LiquidFreeEnergyWithTemp.outputs.free_energy)
wf.TransitionTemperature = TransitionTemperature(phase1=wf.Solid, phase2=wf.Liquid, Tmin=800, Tmax=1000)

In [10]:
pf = PyironFlow([wf], nodes_path="pyiron_nodes")

In [11]:
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Need the solution?</b></p>
Copy paste the following code into a cell and execute:
</br>
</br>
pf = PyironFlow([Workflow("calphy-tm1")]) </br>
pf.gui

</div>

## <font style="font-family:roboto;color:#455e6c"> Calculating a binary phase diagram </font> 

Now that we have seen the thermodynamic functions in Landau, we will make a simple eutectic binary phase diagram with Landau.

In [12]:
from pyiron_nodes.thermodynamics.landau import LinePhase, TemperatureLinePhase, IdealSolution, CalcPhaseDiagram, PlotConcPhaseDiagram, PlotMuPhaseDiagram, PhasesFromDataFrame
from pyiron_nodes.basic.list import List5
from pyiron_nodes.basic.math import Linspace

In [13]:
wf = Workflow("landau_toy")

In [14]:
wf.LinePhase0 = LinePhase(name="SolidA", concentration=0, energy=2.45, entropy=0.00001)
wf.LinePhase1 = LinePhase(name="SolidB", concentration=1, energy=2.95, entropy=0.00001)
wf.LinePhase2 = LinePhase(name="LiquidA", concentration=0, energy=2.5, entropy=0.0001)
wf.LinePhase3 = LinePhase(name="LiquidB", concentration=1, energy=3.1, entropy=0.0002)
wf.IdealSolution = IdealSolution(name="liquid", phase1=wf.LinePhase2, phase2=wf.LinePhase3)
wf.Linspace = Linspace(x_min=250, x_max=1000, num_points=25)
wf.List5 = List5(x1=wf.LinePhase0, x2=wf.LinePhase1, x3=wf.IdealSolution)
wf.CalcPhaseDiagram = CalcPhaseDiagram(phases=wf.List5, temperatures=wf.Linspace, chemical_potentials=50)
wf.PlotConcPhaseDiagram = PlotConcPhaseDiagram(phase_data=wf.CalcPhaseDiagram, plot_tielines=True)

In [15]:
pf = PyironFlow([wf])
pf.gui

## <font style="font-family:roboto;color:#455e6c"> MgCa phase diagram </font> 

Now we will take the same approach to calculate a phase diagram with real data, in this case the MgCa system. The data can be found [here](https://github.com/eisenforschung/mgalca-mtp-data).

In [16]:
from pyiron_nodes.basic.file import ReadDataFrame

In [17]:
def make_phase(dd, temperature_parameters, concentration_parameters):
    name = dd.phase.iloc[0]
    # minus 2 for terminals
    # minus 1 to be not exactly interpolating
    sub = [landau.phases.TemperatureDependentLinePhase(
                f'{row.phase}_{c:.03}', c, 
                row.temperature, row.free_energy, 
                interpolator=landau.interpolate.SGTE(temperature_parameters)
            ) for c, row in dd.set_index('composition').iterrows()]
    # only a single concentration
    if len(sub) == 1:
        return replace(sub[0], name=name)
    if concentration_parameters is not None:
        interp_params = min(len(dd)-2-1, concentration_parameters)
        # terminals are present
        if len({0, 1}.intersection([s.line_concentration for s in sub]))==2:
            if len(sub) == 2: # only terminals are present
                return landau.phases.IdealSolution(name, *sub)
            else:
                return landau.phases.RegularSolution(name, sub, interp_params)
        else:
            return landau.phases.InterpolatingPhase(name, sub, interp_params, num_samples=1000)
    else:
        return sub

In [18]:
wf = Workflow("landau_MgCa")

In [19]:
wf.ReadDataFrame = ReadDataFrame(filename="data/MgCaFreeEnergies.pckl.gz", compression="gzip")
wf.PhasesFromDataFrame = PhasesFromDataFrame(dataframe=wf.ReadDataFrame)
wf.Linspace = Linspace(x_min=200, x_max=1200, num_points=50)
wf.CalcPhaseDiagram = CalcPhaseDiagram(phases=wf.PhasesFromDataFrame.outputs.phase_list, temperatures=wf.Linspace, chemical_potentials=50)
wf.PlotConcPhaseDiagram = PlotConcPhaseDiagram(phase_data=wf.CalcPhaseDiagram, plot_tielines=True)
wf.PlotMuPhaseDiagram = PlotMuPhaseDiagram(phase_data=wf.CalcPhaseDiagram)

In [20]:
pf = PyironFlow([wf], nodes_path="pyiron_nodes")
pf.gui

---
---

### <font style="color:#455e6c" face="Helvetica" > Calculation of free energies: Thermodynamic integration </font>

<img src="img/fig1.png" width="1000">

- free energy of reference system are known: Einstein crystal, [Uhlenbeck-Ford model](https://doi.org/10.1063/1.4967775)
- the two systems are coupled by 
$$
H(\lambda) = \lambda H_f + (1-\lambda)\lambda H_i
$$
- Run calculations for each $\lambda$ and integrate 
$$
G_f = G_i + \int_{\lambda=0}^1 d\lambda \bigg \langle  \frac{\partial H(\lambda)}{\partial \lambda } \bigg \rangle
$$

To calculate $F$,

- for each phase
    - for each pressure
        - for each temperature
            - for each $\lambda$

If we choose 100 different $\lambda$ values; 100 calculations are needed for each temperature and pressure! 

**Dimensionality: (phase, $P$, $T$, $\lambda$)**

### <font style="color:#455e6c" face="Helvetica" > Speeding things up: Non-equilibrium calculations </font>

##### Non-Equilibrium Hamiltonian Interpolation

<img src="img/cimg5.png" width="600">

In this method:

- Discrete $\lambda$ parameter is replaced by a time dependent $\lambda(t)$
- Instead of running calculations at each $\lambda$, run a single, short, non-equilibrium calculation

$$
G_f = G_i + \int_{t_i}^{t_f} dt \frac{d\lambda (t)}{dt}  \frac{ H(\lambda)}{\partial \lambda }
$$

As discussed:
- the coupling parameter $\lambda$ earlier is replaced by a time dependent parameter
- The equation also no longer has an ensemble average  

These aspects makes it quite easy and fast to estimate this integral.

However:
- this equation holds when the switching betwen the system of interest and reference system is carried out infinitely slowly
- Practically, this is not not possible. 

Therefore we can write:

$$
\Delta G = W_{rev} = W_s - E_d
$$

$$
W_s = \int_{t_i}^{t_f} dt \frac{d\lambda (t)}{dt}  \frac{ H(\lambda)}{\partial \lambda }
$$

- $E_d$ is the energy dissipation
- $E_d \to 0$ when $t_f-t_i \to \infty$

So far, so good, but how is this useful?

- Instead of a single transformation from system of interest to reference, we switch back too
- These are called forward $(i \to f)$ and backward $(f \to i)$ switching
- $t_f - t_i = t_{sw}$ is the switching time in each direction
- If $t_{sw}$ is long enough, $E_d^{i \to f} = E_d^{f \to i}$
- and $\Delta G = \frac{1}{2} (W_s^{i \to f} - W_s^{f \to i})$

Now, we have all the components required for actual calculations.

We have also managed to successfully reduce the dimensionality

- for each phase
    - for each pressure
        - for each temperature
            - ~~for each $\lambda$~~

**Dimensionality: (phase, $P$, $T$)**


So, how do we calculate the free energy of a system modelled with a given interatomic potential?

### <font style="color:#455e6c" face="Helvetica" > Free energy for solids </font>

#### Task: Find free energy of Al in FCC lattice at 500 K and 0 pressure

1. Create an Al FCC lattice
2. Choose an interatomic potential
3. Run MD calculations at 500 K and 0 pressure to equilibrate the system
4. Introduce the reference system
5. Switch....
6. .....

Steps 1-3 should be fairly easy, we saw this in the last days and also in the first session. But how do we introduce a reference system?

- A reference system is simply one for which the free energy is analytically known ($G_i$)
- We calculate the free energy difference between this and the system of interest.

In case of solids, a good choice of a reference system is the Einstein crystal. An Einstein crystal is a set of independent harmonic oscillators attached to the lattice positions. 


The free energy of the Einstein crystal is:

$$
F_E = 3 k_B T \sum_{i} ln \bigg ( \frac{\hbar \omega_i}{k_B T} \bigg )
$$

We need to calculate:

- $\omega$
- A common way is $$  \frac{1}{2} k_i \langle (\Delta \pmb{r}_i)^2 \rangle = \frac{3}{2} k_\mathrm{B} T $$
- $\langle (\Delta \pmb{r}_i)^2 \rangle$ is the mean squared displacement.

Now that we know about the reference system, let's update our pseudo workflow:

1. Create an Al fcc lattice
2. Choose an interatomic potential
3. Run MD calculations at 500 K and 0 pressure to equilibrate the system
4. Calculate the mean squared displacement, therefore spring constants
5. Switch system of interest to reference system
6. Equilibrate the system
7. Switch back to system of interest
8. Find the work done
9. Add to the free energy of the Einstein crystal

As you can see, there are a number of steps that need to be done. This is where **calphy** and **pyiron** come in. These tools automatise all of the above steps and makes it easy to perform these calculations.

### <font style="color:#455e6c" face="Helvetica" > Free energy as function of temperature </font>

<img src="img/cimg6.png" width="600">

Gibb's free energy via reversible scaling at a constant pressure is given by,

$ G(N,P,T_f) = G(N,P,T_i) + \dfrac{3}{3}Nk_BT_f\ln{\dfrac{T_f}{T_i}} + \dfrac{T_f}{T_i}\Delta G $,

Therefore, $G(N,P,T_f)$ can be computed from $G(N,P,T_i)$ via the free energy difference $\Delta G$. 

Here, $\Delta G = \dfrac{1}{2}[W_{if}-W_{fi}$] --- (2)

The reversible work is related to the internal energy $U$ by,
$W = \int_{1}^{\lambda_f}<U> \,d\lambda$ --- (3)

Using MD $W$ can be computed as:
- equilibrate for time $t_{eq}$ in NPT ensemble
- switch $\lambda$ : $1->\dfrac{T_f}{T_i}$ over time $t_{sw}$
- calculate work $W_{if}$ from (3)
- equilibrate for time $t_{eq}$ in NPT ensemble
- switch $\lambda$ : $\dfrac{T_f}{T_i}->1$ over time $t_{sw}$
- calculate work $W_{fi}$ from (3).

### <font style="color:#455e6c" face="Helvetica" > Free energy for liquids </font>

**How is the liquid prepared in this calculation?**

- Start from the given structure
- This structure is heated until it melts.
- Melting of the structure is automatically detected by calphy
- Once melted, it is equilibrated to the required temperature and pressure.

**What about the reference system for liquid?**

The reference system for the Liquid structure is also different. In this case, the Uhlenbeck-Ford system is used as the reference system for liquid.

The Uhlenbeck-Ford model is described by,

$$
E = - \epsilon \log(1-\exp(-r^2/\sigma^2))
$$

where,

$$
\epsilon = p k_B T
$$

$\epsilon$ and $\sigma$ are energy and length scales, respectively.

It is purely repulsive liquid model which does not undergo any phase transformations.

### <font style="color:#455e6c" face="Helvetica" > Comparison of melting temperature methods </font>

<img src="img/tm_methods.png" width="900">

### <font style="color:#455e6c" face="Helvetica" > Further reading </font>
- [Menon et al. "From electrons to phase diagrams with machine learning potentials using pyiron based automated workflows." npj Computational Materials 10, 2024](https://www.nature.com/articles/s41524-024-01441-0)
- [Menon, Sarath, Yury Lysogorskiy, Jutta Rogal, and Ralf Drautz. “Automated Free-Energy Calculation from Atomistic Simulations.” Physical Review Materials 5, no. 10 (October 11, 2021): 103801.](https://doi.org/10.1103/PhysRevMaterials.5.103801).
- [Freitas, Rodrigo, Mark Asta, and Maurice de Koning. “Nonequilibrium Free-Energy Calculation of Solids Using LAMMPS.” Computational Materials Science 112 (February 2016): 333–41.](https://doi.org/10.1016/j.commatsci.2015.10.050).
- [Paula Leite, Rodolfo, and Maurice de Koning. “Nonequilibrium Free-Energy Calculations of Fluids Using LAMMPS.” Computational Materials Science 159 (March 2019): 316–26.](https://doi.org/10.1016/j.commatsci.2018.12.029).
- [Koning, Maurice de, A. Antonelli, and Sidney Yip. “Optimized Free-Energy Evaluation Using a Single Reversible-Scaling Simulation.” Physical Review Letters 83, no. 20 (November 15, 1999): 3973–77.](https://doi.org/10.1103/PhysRevLett.83.3973).
- [Paula Leite, Rodolfo, Rodrigo Freitas, Rodolfo Azevedo, and Maurice de Koning. “The Uhlenbeck-Ford Model: Exact Virial Coefficients and Application as a Reference System in Fluid-Phase Free-Energy Calculations.” The Journal of Chemical Physics 145, no. 19 (November 21, 2016): 194101.](https://doi.org/10.1063/1.4967775).



### <font style="font-family:roboto;color:#455e6c"> Software used in this notebook </font>  

- [calphy](https://calphy.org)
- [landau](https://github.com/eisenforschung/landau)
- [LAMMPS](https://www.lammps.org/)
- [pyiron](https://pyiron.org/)
- [pyiron_workflow](https://github.com/pyiron/pyiron_workflow)

<img src="img/logo_roll.png" width="1200">